# Module 5.2: Data Pipelines & Orchestration

**Duration:** 38 minutes  
**Level:** 2 (Production Data Management)  
**Prerequisites:** Level 1 M1.3 (Document Processing), M5.1 (Incremental Indexing)

---

## Introduction & Hook

### The Problem

In M5.1, you built incremental indexing that detects changed documents and updates only what's necessary. It works beautifully... **when you remember to run it**.

But here's what's happening in production right now:
- It's 2 AM
- Your compliance documents updated three hours ago
- Your RAG system is still serving yesterday's stale data
- Nobody manually triggered the refresh

Tomorrow morning, your legal team will ask questions about the new policy updates. Your RAG will give outdated answers. Someone will notice. **Trust in your system takes a hit.**

### The Solution

The problem isn't your incremental indexing—that works. The problem is you're treating data refresh like a manual chore when it needs to be an **automated, reliable pipeline**.

**You need orchestration.**

### What You'll Learn

By the end of this notebook, you'll be able to:
- ✅ Schedule automated data refresh pipelines that run daily, hourly, or on-demand
- ✅ Implement parallel processing that cuts refresh time from 40 minutes to 8 minutes for 5,000 documents
- ✅ Handle pipeline failures gracefully with automatic retries and alerting
- ✅ Monitor pipeline health with metrics that show you exactly where bottlenecks are
- ✅ **Critical:** Understand when Airflow is overkill (hint: it is for most side projects) and what simpler alternatives exist

In [ ]:
# Setup and imports
import os
import sys
from datetime import datetime

# Import our module functions
from l2_m2_datapipelines_orchestration import (
    detect_changed_documents,
    chunk_document,
    embed_chunks,
    process_document_batch,
    upsert_to_pinecone,
    handle_deletions,
    run_incremental_refresh_pipeline
)

from config import (
    DATA_DIR,
    CHECKSUM_FILE,
    get_clients,
    validate_config,
    BATCH_SIZE,
    MAX_WORKERS
)

print("✓ Imports successful")
print(f"Data directory: {DATA_DIR}")
print(f"Checksum file: {CHECKSUM_FILE}")

In [ ]:
# Validate configuration
config_valid = validate_config()

if config_valid:
    print("✓ Configuration valid - API calls enabled")
else:
    print("⚠️ Configuration incomplete - Running in demo mode (no API calls)")
    print("To enable API calls, copy .env.example to .env and add your keys")

# Get clients (may be None if keys missing)
clients = get_clients()
print(f"\nOpenAI Client: {'✓ Ready' if clients['openai'] else '✗ Not configured'}")
print(f"Pinecone Client: {'✓ Ready' if clients['pinecone'] else '✗ Not configured'}")

---

## Section 2: Prerequisites & Setup

### Starting Point Verification

Your system currently has:

**From M5.1:**
- ✅ `detect_changes()` function that returns list of modified documents
- ✅ `update_document()` function that re-chunks and updates vectors
- ✅ File checksum storage for change detection
- ✅ Manual execution: you run `python incremental_update.py` when needed

**The limitation:** This is manual and single-threaded.

### Current Approach (M5.1)

```python
# Your current M5.1 approach
changed_docs = detect_changes('/data/documents')
for doc_path in changed_docs:
    update_document(doc_path)  # Processes one at a time

# Problems:
# - For 5,000 documents, this takes 40+ minutes
# - Runs only when you manually execute
# - If it crashes at document 3,247, you restart from scratch
```

### Target Architecture (After Today)

By the end, this will become:
- ✅ Automated scheduling (runs daily at 2 AM, no human involved)
- ✅ Parallel processing (8 minutes for 5,000 documents)
- ✅ Checkpointing (resume from failure, not restart)
- ✅ Monitored (Slack alert if pipeline fails)

In [ ]:
# Demonstrate change detection (core M5.1 functionality)
print("=== Testing Change Detection ===\n")

result = detect_changed_documents(DATA_DIR, CHECKSUM_FILE)

print(f"Changed files: {len(result['changed_files'])}")
print(f"Deleted files: {len(result['deleted_files'])}")

if result['changed_files']:
    print(f"\nFirst changed file: {result['changed_files'][0]}")

# Expected: On first run, all files are "new"
# Expected: On subsequent runs, only actually changed files appear

---

## Section 3: Theory Foundation - How Airflow Works

### What Is Apache Airflow?

Apache Airflow is a **workflow orchestration platform**. Think of it as a project manager for your data tasks.

**What Airflow does:**
- 📅 **Scheduling:** Runs tasks at specific times (daily at 2 AM, hourly, etc.)
- 🔗 **Dependency Management:** Task B waits for Task A to finish
- ⚡ **Parallelization:** Distributes work across multiple workers
- 🔄 **Failure Recovery:** Retries individual tasks, not entire pipelines
- 👀 **Monitoring:** Visibility into what's running, what failed, what succeeded

**What Airflow does NOT do:**
- ❌ Make your code faster (it orchestrates, doesn't optimize)
- ❌ Handle real-time streaming (it's batch-oriented)
- ❌ Replace your business logic (you still write the processing code)

### Core Architecture

```
┌─────────────────────────────────────────────────────┐
│                    DAG DEFINITION                   │
│  (Your Python file describing workflow)             │
│                                                      │
│  detect_changes → chunk_documents → embed → upsert  │
│        │                                      ▲      │
│        └──────── (depends on) ────────────────┘     │
└──────────────────┬──────────────────────────────────┘
                   │
                   ▼
┌─────────────────────────────────────────────────────┐
│                 EXECUTOR (Celery)                   │
│  (Distributes tasks to workers)                     │
└──────────────────┬──────────────────────────────────┘
                   │
        ┌──────────┼──────────┬──────────┐
        ▼          ▼          ▼          ▼
    Worker 1   Worker 2   Worker 3   Worker 4
    (Process  (Process   (Process   (Process
     Doc 1-    Doc 501-   Doc 1001-  Doc 1501-
     500)      1000)      1500)      2000)
```

### Key Concepts

1. **DAG (Directed Acyclic Graph):** Your workflow definition
   - Directed: Tasks have a defined order
   - Acyclic: No circular dependencies
   - Graph: Visual representation of task relationships

2. **Task:** A unit of work (detect changes, embed documents, etc.)

3. **Operator:** Template for a task (PythonOperator, BashOperator)

4. **Scheduler:** Monitors DAGs and triggers execution when conditions are met

5. **Executor:** Determines how tasks run (sequential vs. parallel)

6. **XCom:** Cross-communication mechanism for passing data between tasks

### Common Misconception

**WRONG:** "Airflow makes my code faster."

**CORRECT:** "Airflow doesn't magically speed up your document processing function. What it does is:
- Let you run multiple instances in parallel (faster overall)
- Prevent re-running already-successful tasks (faster recovery)
- Give you visibility into bottlenecks (faster debugging)

Your actual `embed_document()` function still takes the same time. But instead of processing 5,000 documents one-by-one in 40 minutes, you can process them 500-at-a-time across 10 workers in 8 minutes."

In [ ]:
# Demonstrate pipeline stages
import time

print("=== Pipeline Stage Demonstration ===")

# Stage 1: Detect changes
print("\nStage 1: Detecting changes...")
start = time.time()
changes = detect_changed_documents(DATA_DIR, CHECKSUM_FILE)
print(f"  ✓ Found {len(changes['changed_files'])} changed files ({time.time()-start:.2f}s)")

# Stage 2: Chunk a document (if we have files)
if changes['changed_files']:
    print("\nStage 2: Chunking document...")
    start = time.time()
    sample_file = changes['changed_files'][0]
    chunks = chunk_document(sample_file, chunk_size=512)
    print(f"  ✓ Created {len(chunks)} chunks ({time.time()-start:.2f}s)")
    print(f"  First chunk preview: {chunks[0]['text'][:80]}...")
else:
    print("\nStage 2: No new files to chunk")

# Expected: Sequential execution of stages
# Expected: Each stage completes before next begins

---

## Section 4: Hands-on Implementation

### Step 1: Document Processing Pipeline

Let's build the core pipeline step-by-step. In a production Airflow setup, each of these would be a separate task in a DAG.

**Pipeline stages:**
1. Detect changed documents (using checksums)
2. Process documents (chunk and embed)
3. Upsert to vector database
4. Handle deletions

### Processing Documents in Batches

The key to efficient pipelines is batch processing. Instead of processing one document at a time, we process batches.

In [ ]:
# Process a batch of documents
print("=== Batch Processing Demonstration ===")

# Get changed files
changes = detect_changed_documents(DATA_DIR, CHECKSUM_FILE)
changed_files = changes['changed_files']

if changed_files:
    print(f"\nProcessing {len(changed_files)} documents...\n")
    
    # Process batch (embeddings skipped if no OpenAI client)
    processed = process_document_batch(
        changed_files,
        openai_client=clients['openai'],  # May be None
        chunk_size=512,
        chunk_overlap=50
    )
    
    print(f"✓ Processed {len(processed)} documents")
    
    if processed:
        total_chunks = sum(len(p['chunks']) for p in processed)
        print(f"✓ Total chunks created: {total_chunks}")
        print(f"✓ Embeddings generated: {len(processed[0].get('embeddings', []))} per doc")
else:
    print("\nNo changed files to process")

# Expected: All documents processed
# Expected: If no API keys, embeddings list is empty (graceful degradation)

---

## Section 5: Reality Check (TVH Framework v2.0)

### What This DOESN'T Do

**Important:** Airflow orchestrates pipelines, but it doesn't solve all problems.

**Airflow does NOT:**
- ❌ Make your code faster (it parallelizes, not optimizes)
- ❌ Handle real-time requirements (<1 minute latency)
- ❌ Replace proper error handling in your code
- ❌ Automatically scale to infinite documents
- ❌ Work well on Windows (limited support)
- ❌ Run efficiently with <4GB RAM

### Trade-offs You Accepted

By choosing Airflow, you accepted:

1. **Operational Complexity:** You now manage a scheduler, executor, workers, and a database
2. **Infrastructure Overhead:** Minimum 4GB RAM, PostgreSQL in production
3. **Learning Curve:** Understanding DAGs, XCom, operators, executors
4. **Batch-Only Processing:** No streaming, no sub-minute latency
5. **Cost:** ~$150/month for production setup (compute + database + monitoring)

### When This Approach Breaks

**Scenario 1: Rate Limits**
```python
# Problem: OpenAI has 3,500 RPM limit
# With 10 workers and 10,000 documents
# You need 50,000 embeddings → 14 minutes minimum
# Parallelization can't go faster than API rate limits
```

**Scenario 2: Memory Exhaustion**
```python
# Problem: Loading all documents into memory at once
# Fix: Process in batches, stream data
ERROR - MemoryError: Cannot allocate memory
```

**Scenario 3: Pipeline Deadlock**
```python
# Problem: Database connection pool exhaustion
# Fix: Limit connections, use connection pooling
# Set max_active_runs=1 in DAG config
```

**Scenario 4: Silent Failures**
```python
# Problem: Pipeline reports success but data is incorrect
# Fix: Add validation checks, row count reconciliation
# Symptom: Embedding dimension mismatch
```

In [ ]:
# Demonstrate full pipeline with error handling
print("=== Full Pipeline Execution ===")

result = run_incremental_refresh_pipeline(
    documents_path=DATA_DIR,
    checksums_file=CHECKSUM_FILE,
    openai_client=clients['openai'],
    pinecone_index=clients['pinecone'],
    max_workers=1  # Sequential for demo
)

print(f"\n=== Pipeline Summary ===")
print(f"Status: {result['status']}")
print(f"Duration: {result['duration_seconds']}s")
print(f"Files changed: {result.get('files_changed', 0)}")
print(f"Files processed: {result.get('files_processed', 0)}")
print(f"Vectors upserted: {result.get('vectors_upserted', 0)}")
print(f"Deletions handled: {result.get('deletions_handled', 0)}")

# Expected: Pipeline completes successfully
# Expected: If no API keys, vectors_upserted = 0 (graceful skip)

---

## Section 6: Alternative Solutions (TVH Framework v2.0)

Airflow isn't the only way to orchestrate pipelines. Here are alternatives:

### Alternative 1: Simple Cron Jobs

**Best for:** <500 documents, solo developer, simple pipelines

**Pros:**
- Zero infrastructure overhead
- No learning curve
- Free

**Cons:**
- No retry logic
- No visibility/monitoring
- No parallelization
- No dependency management

**Example:**
```bash
# Add to crontab: Run daily at 2 AM
0 2 * * * cd /app && python incremental_update.py >> logs/cron.log 2>&1
```

**Cost:** $0/month  
**Setup time:** 5 minutes

---

### Alternative 2: Prefect (Modern Airflow Alternative)

**Best for:** Teams wanting modern Python tooling, 1K-50K documents

**Pros:**
- Modern Python API (no DAG complexity)
- Better error handling than Airflow
- Native Kubernetes support
- Managed cloud option

**Cons:**
- Smaller ecosystem than Airflow
- Managed cloud costs $1,200+/year

**Cost:** Self-hosted $0, Cloud $100/month  
**Setup time:** 30 minutes

---

### Alternative 3: AWS EventBridge + Lambda (Cloud-Native)

**Best for:** AWS-native shops, event-driven architecture

**Pros:**
- Serverless (no infrastructure management)
- Scales automatically
- Pay-per-use pricing

**Cons:**
- AWS vendor lock-in
- Cold start latency (1-3 seconds)
- 15-minute Lambda timeout limit

**Cost:** ~$30/month for 10K documents  
**Setup time:** 2 hours

---

### Alternative 4: Kafka + Stream Processing (Event-Driven)

**Best for:** Real-time requirements, >10K daily updates

**Pros:**
- True real-time processing (<1 second latency)
- Handles high-throughput event streams
- Decouples producers/consumers

**Cons:**
- High operational complexity
- Requires dedicated team
- Expensive ($220+/month minimum)

**Cost:** $220+/month  
**Setup time:** 1-2 weeks

---

### Decision Framework

| Scale | Frequency | Complexity | Recommended Solution |
|-------|-----------|------------|---------------------|
| <500 docs | Daily | Simple | **Cron job** |
| 500-5K docs | Daily/Hourly | Medium | **Airflow** or **Prefect** |
| 5K-50K docs | Hourly | High | **Airflow** with Celery |
| 50K+ docs | Real-time | Very High | **Kafka + Spark** |
| AWS-only | Any | Any | **EventBridge + Lambda** |

---

### Cost Breakdown (Monthly) for 10K Documents

**Airflow (Self-Hosted):**
- Compute (4 workers, 2GB each): $30
- PostgreSQL database: $12
- Embedding API (OpenAI): $50
- Pinecone vectors: $70
- Monitoring (Prometheus): $12
- **Total: ~$162/month**

**Prefect Cloud:**
- Managed Prefect: $100
- Embedding API: $50
- Pinecone: $70
- **Total: ~$220/month**

**Cron Job:**
- Embedding API: $50
- Pinecone: $70
- **Total: ~$120/month** (no orchestration costs)

In [ ]:
# Decision helper: Should you use Airflow?
def should_use_airflow(num_docs, update_frequency_hours, has_dependencies, team_size):
    """
    Decision helper based on TVH Framework criteria.
    
    Returns recommendation and reasoning.
    """
    reasons = []
    
    # Check scale
    if num_docs < 500:
        return "Use cron job", ["Scale too small for Airflow overhead"]
    
    # Check real-time requirements
    if update_frequency_hours < 0.016:  # <1 minute
        return "Use event-driven (Kafka)", ["Real-time latency requirements"]
    
    # Check team capacity
    if team_size < 2:
        reasons.append("Small team - consider Prefect for easier maintenance")
    
    # Check dependencies
    if not has_dependencies:
        reasons.append("No dependencies - simple cron may suffice")
    
    if num_docs >= 500 and num_docs < 50000:
        recommendation = "Use Airflow"
        reasons.append(f"Scale ({num_docs} docs) justifies orchestration")
        reasons.append(f"Update frequency ({update_frequency_hours}h) is batch-friendly")
    elif num_docs >= 50000:
        recommendation = "Use Airflow + Celery or Spark"
        reasons.append("Large scale requires distributed processing")
    else:
        recommendation = "Use cron job"
    
    return recommendation, reasons

# Test scenarios
print("=== Airflow Decision Helper ===")

scenarios = [
    ("Side project", 200, 24, False, 1),
    ("Small startup", 2000, 6, True, 3),
    ("Enterprise", 25000, 1, True, 10),
    ("Real-time app", 5000, 0.01, True, 5),
]

for name, docs, freq, deps, team in scenarios:
    rec, reasons = should_use_airflow(docs, freq, deps, team)
    print(f"\n{name}: {docs} docs, {freq}h frequency, team={team}")
    print(f"  → {rec}")
    for r in reasons:
        print(f"    • {r}")

---

## Section 7: When NOT to Use Airflow (Critical)

### Scenario 1: Side Projects and MVPs (<500 documents, solo developer)

**Don't use Airflow if:**
- You have <500 documents
- You're a solo developer
- Updates happen daily or less frequently
- You don't need parallel processing

**Use instead:** Cron job with logging

---

### Scenario 2: Real-Time Requirements (<1 minute latency needed)

**Don't use Airflow if:**
- You need sub-minute response times
- Documents update continuously
- You need streaming processing

**Use instead:** Event-driven architecture (Kafka + stream processing)

---

### Scenario 3: Simple Linear Pipelines (No Dependencies)

**Don't use Airflow if:**
- Your pipeline is: read → process → write (no branches)
- Tasks don't have complex dependencies
- No need for retries or monitoring

**Use instead:** Standard Python script with error logging

---

### Scenario 4: Windows Environments

**Don't use Airflow if:**
- You're restricted to Windows
- Can't use Docker/WSL2

**Use instead:** Prefect (better Windows support) or cloud-native solutions

---

### Scenario 5: Resource-Constrained (<4GB RAM available)

**Don't use Airflow if:**
- Available RAM <4GB
- Can't run PostgreSQL
- Limited CPU cores

**Use instead:** Serverless (AWS Lambda + EventBridge)

---

## Section 8: Common Production Failures

### Failure 1: Pipeline Deadlock with Parallel Processing

**Symptom:** Tasks hang indefinitely, workers appear stuck

**Cause:** Database connection pool exhaustion

**Fix:**
```python
# Limit connection pool size
from sqlalchemy import create_engine
engine = create_engine(DB_URL, pool_size=10, max_overflow=20)

# Set task timeout
task = PythonOperator(
    task_id='process',
    execution_timeout=timedelta(minutes=30)
)
```

---

### Failure 2: Task Scheduling Conflicts (Overlapping Runs)

**Symptom:** Multiple pipeline instances running simultaneously

**Cause:** Previous run didn't complete before next scheduled run

**Fix:**
```python
dag = DAG(
    'rag_refresh',
    max_active_runs=1,  # Only one instance at a time
    catchup=False  # Don't backfill
)
```

---

### Failure 3: Resource Exhaustion (Memory)

**Symptom:** `MemoryError: Cannot allocate memory`

**Cause:** Loading all documents into memory at once

**Fix:**
```python
# Process in batches, don't accumulate
for batch in chunks(files, batch_size=100):
    process_batch(batch)
    # Batch goes out of scope, memory freed
```

---

### Failure 4: Zombie Processes

**Symptom:** Orphaned processes consuming resources

**Cause:** No cleanup in finally blocks

**Fix:**
```python
import signal

try:
    process_documents()
finally:
    # Cleanup: kill process group
    os.killpg(os.getpgid(0), signal.SIGTERM)
```

---

### Failure 5: Silent Failures (No Validation)

**Symptom:** Pipeline reports success but data is incorrect

**Cause:** Swallowing exceptions, no validation

**Fix:**
```python
# Add validation
def validate_embeddings(embeddings):
    assert len(embeddings) > 0, "No embeddings generated"
    assert len(embeddings[0]) == 1536, "Dimension mismatch"
    return True
```

---

## Section 9: Production Considerations

### Monitoring Requirements

**Critical metrics to track:**
1. Task execution duration (baseline vs. current)
2. Error rate percentage (alert if >10% sustained)
3. Worker availability (alert if down >2 minutes)
4. Data freshness lag (alert if >2 hours behind)
5. Queue depth (alert if backlog growing)

### Alert Thresholds

**Critical alerts (immediate):**
- Pipeline failure
- Error rate >25% for 5+ minutes
- All workers down

**Warning alerts:**
- Pipeline duration >1.5× expected
- Error rate >10% for 5+ minutes
- Data freshness lag >2 hours

---

## Section 10: Decision Card

### Use Airflow When:

✅ You have 500-50,000 documents  
✅ Updates happen hourly or daily (batch-friendly)  
✅ You have complex task dependencies  
✅ You need parallel processing  
✅ You have operational capacity (2+ person team)  
✅ You can allocate 4+ GB RAM  
✅ You need monitoring and alerting  

### Don't Use Airflow When:

❌ You have <500 documents (use cron)  
❌ You need real-time (<1 min latency, use Kafka)  
❌ Simple linear pipeline (use Python script)  
❌ Windows-only environment (use Prefect)  
❌ <4GB RAM available (use serverless)  
❌ Solo developer side project (too much overhead)  

### The Bottom Line

**Airflow is heavyweight infrastructure.** It's the right tool for production data pipelines at scale, but it's overkill for most side projects.

**Ask yourself:**
1. Do I really need orchestration, or just scheduling? (If just scheduling → cron)
2. Do I have the operational capacity to maintain it? (If no → managed service)
3. Is my scale large enough to justify the complexity? (If <500 docs → no)

**Remember:** The best tool is the simplest one that meets your requirements.


---

## Summary & Next Steps

### What You Learned

✅ **Change Detection:** Using checksums to identify modified documents  
✅ **Batch Processing:** Chunking and embedding documents efficiently  
✅ **Pipeline Architecture:** Understanding Airflow's DAG, tasks, and executors  
✅ **Error Handling:** Graceful degradation when services unavailable  
✅ **Reality Check:** When Airflow is overkill and alternatives exist  
✅ **Common Failures:** Production issues and how to fix them  
✅ **Cost Awareness:** ~$162/month for 10K documents  

### What We Built

This notebook demonstrated:
1. Incremental indexing with change detection
2. Document chunking and embedding pipeline
3. Vector database upsertion
4. Graceful degradation (works without API keys)
5. Decision framework for choosing orchestration tools

### Production Deployment

To deploy to production:

1. **Set up Airflow:**
```bash
# Install Airflow
pip install apache-airflow

# Initialize database
airflow db init

# Create DAG from this code
# Place in ~/airflow/dags/
```

2. **Configure environment variables** (see `.env.example`)

3. **Set up monitoring** (Prometheus + Grafana)

4. **Test with small dataset** before full deployment

### Alternative: Use the FastAPI Endpoint

For simpler use cases, run the included FastAPI app:

```bash
# Start the API server
python app.py

# Trigger refresh via API
curl -X POST http://localhost:8000/refresh
```

### Resources

- Apache Airflow Docs: https://airflow.apache.org/
- Prefect Docs: https://docs.prefect.io/
- Source code: `l2_m2_datapipelines_orchestration.py`
- Full README: See `README.md` for setup instructions

### Next Module

**Module 5.3:** Advanced Monitoring & Observability
- Distributed tracing for RAG pipelines
- Cost tracking and optimization
- Performance profiling at scale


In [ ]:
# Final check: Verify all components work
print("=== Final System Check ===")

print("\n1. Configuration:")
print(f"   {'✓' if validate_config() else '⚠'} Config valid")

print("\n2. Clients:")
clients = get_clients()
print(f"   {'✓' if clients['openai'] else '✗'} OpenAI client")
print(f"   {'✓' if clients['pinecone'] else '✗'} Pinecone client")

print("\n3. Data:")
import os
doc_count = len([f for f in os.listdir(DATA_DIR) if f.endswith(('.txt', '.pdf', '.docx'))])
print(f"   ✓ Found {doc_count} documents in {DATA_DIR}")

print("\n4. Functions:")
print("   ✓ detect_changed_documents")
print("   ✓ chunk_document")
print("   ✓ embed_chunks")
print("   ✓ process_document_batch")
print("   ✓ upsert_to_pinecone")
print("   ✓ handle_deletions")
print("   ✓ run_incremental_refresh_pipeline")

print("\n=== System Ready ===")
print("\nTo enable full functionality, add API keys to .env file")
print("To run via API: python app.py")
print("To run tests: pytest tests_smoke.py")